# python Library

In [ ]:
## usual impport functions : general use
import numpy as np
from numpy import cosh, zeros_like, mgrid, zeros
import matplotlib.pyplot as plt
import pandas as pd
import math

# from interpolation.complete_poly import (CompletePolynomial,
#                                          n_complete, complete_polynomial,
#                                          complete_polynomial_der,
#                                          _complete_poly_impl,
#                                          _complete_poly_impl_vec,
#                                          _complete_poly_der_impl,
#                                          _complete_poly_der_impl_vec)
from numba import jit, vectorize, njit, prange

## scipy functions for optimization (not all used)
from scipy.optimize import root
from scipy.sparse import spdiags, kron
from scipy.sparse.linalg import spilu, LinearOperator
from scipy.optimize import leastsq
from scipy.optimize import minimize
from scipy.special import j1
from scipy.interpolate import interp1d ## 1d interpolation, 2d and radial basis are avaiable for higher dimension interpolation
from numpy.linalg import norm, solve
from scipy.linalg import kron
from scipy.stats import norm
from scipy.linalg import cholesky


from concurrent.futures import ProcessPoolExecutor
from multiprocessing import Pool
import requests
import time

import os
print(f"Available CPUs: {os.cpu_count()}")
### quantecon functions
# import quantecon as qe
# import quantecon_wasm as qe_w
# from quantecon import compute_fixed_point
# from quantecon.markov import DiscreteDP

Available CPUs: 2


In [ ]:

def fnDisclyap(a1, b1, vecflag=False):
    """
    Solve V = sum_{j=0}^∞ a1^j b1 (a1^j)'
    """

    if vecflag:
        n = a1.shape[0]
        I = np.eye(n*n)
        vecV = solve(I - kron(a1, a1), b1.reshape(-1))
        return vecV.reshape(n, n)

    alpha0 = a1.copy()
    gamma0 = b1.copy()

    delta = 5.0
    it = 0

    while delta > np.finfo(float).eps:
        alpha1 = alpha0 @ alpha0
        gamma1 = gamma0 + alpha0 @ gamma0 @ alpha0.T

        delta = np.max(np.abs(gamma1 - gamma0))

        gamma0 = gamma1
        alpha0 = alpha1
        it += 1

        if it > 50:
            raise RuntimeError(f"DiscLyap did not converge: iter={it}, delta={delta}")

    return gamma1

def fnSimulator(initial_state, transition_matrix, sim_time):
    """
    Simulate a Markov chain
    """
    cum_trans = np.cumsum(transition_matrix, axis=1)
    out = np.ones(sim_time, dtype=int)
    out[0] = initial_state

    for t in range(1, sim_time):
        u = np.random.rand()
        out[t] = np.searchsorted(cum_trans[out[t-1]], u)

    return out


def gridMVndx(Ns, Ny):
    Nstar = Ns**Ny
    MVndx = np.zeros((Nstar, Ny), dtype=int)
    uv = np.arange(Ns)

    MVndx[:, 0] = np.tile(uv, Ns**(Ny-1))

    for j in range(1, Ny):
        n1 = Ns**j
        n2 = Ns**(Ny-j-1)
        MVndx[:, j] = np.tile(np.kron(uv, np.ones(n1, dtype=int)), n2)

    return MVndx


def fnTauchen(F, F0, Sigma, Ns=20, bandwidth=3):
    """
    Multivariate Tauchen discretization
    """
    Ny = F.shape[0]
    Nstar = Ns**Ny

    Q = cholesky(Sigma, lower=True)
    iQ = np.linalg.inv(Q)

    EY = np.linalg.solve(np.eye(Ny) - F, F0)
    VarY = fnDisclyap(F, Q @ Q.T)

    Fx = iQ @ F @ Q
    F0x = iQ @ F0
    EX = iQ @ EY
    VarX = iQ @ VarY @ iQ.T
    StdX = np.sqrt(np.diag(VarX))

    # grids
    grids = np.zeros((Ns, Ny))
    Ub = EX + bandwidth * StdX
    Lb = EX - bandwidth * StdX
    steps = (Ub - Lb) / (Ns - 1)

    for i in range(Ny):
        grids[:, i] = np.linspace(Lb[i], Ub[i], Ns)

    MVndx = gridMVndx(Ns, Ny)
    x = grids[MVndx, np.arange(Ny)]
    y = x @ Q.T

    endpoints = grids[:-1] + steps / 2

    P = np.zeros((Nstar, Nstar))

    for s in range(Nstar):
        condmean = x[s] @ Fx.T + F0x
        probby = np.diff(
            np.vstack([
                np.zeros(Ny),
                norm.cdf(endpoints, condmean, 1.0),
                np.ones(Ny)
            ]),
            axis=0
        )
        P[s] = np.prod(probby[MVndx, np.arange(Ny)], axis=1)

    return P, y, x, MVndx



# DMP model

In [ ]:
## steady state - with cobb douglas matching function
## parameters
kap =
gam =
b =
delta =
beta =
alpha =


In [ ]:
def solve_steady_state(pa):
    q = 0.8
    A = 1.0
    err = 1.0

    while err > 1e-10:
        v = (q/pa['m'])**(1/(-pa['xi1'])) / \
            (1 + (1/pa['lambda']) * q * (q/pa['m'])**(1/(-pa['xi1'])))

        n = v*q / pa['lambda']
        u = 1 - n
        theta = v/u
        w = (1-pa['eta'])*pa['b'] + pa['eta']*(A + pa['kappa']*theta)
        c = A*n - pa['kappa']*v + (1-n)*pa['b']

        q_new = pa['kappa'] / (pa['beta']*(A - w + (1-pa['lambda'])*pa['kappa']/q))
        err = abs(q - q_new)
        q = 0.9*q + 0.1*q_new

    return dict(n=n, u=u, q=q, v=v, w=w, c=c)


In [ ]:
EE = (tempq - vq) / vq
logEE = np.log(np.abs(EE[BURNIN:-BURNIN]))

plt.figure()
plt.scatter(range(len(logEE)), logEE)
plt.xlabel("Time")
plt.ylabel("Euler equation error (log)")
plt.show()


In [ ]:
## transistional dynamics

# Julia library

In [2]:
import Pkg

In [3]:
Pkg.add("Debugger")
Pkg.add("JLD")
Pkg.add("Optim")

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed CodeTracking ───── v3.0.0
   Installed JuliaInterpreter ─ v0.10.9
   Installed Highlights ─────── v0.5.3
   Installed Debugger ───────── v0.7.16
    Updating `~/.julia/environments/v1.11/Project.toml`
  [31a5f54b] + Debugger v0.7.16
    Updating `~/.julia/environments/v1.11/Manifest.toml`
  [da1fd8a2] + CodeTracking v3.0.0
  [31a5f54b] + Debugger v0.7.16
  [eafb193a] + Highlights v0.5.3
  [aa1ae85d] + JuliaInterpreter v0.10.9
Precompiling project...
   6030.4 ms  ✓ CodeTracking
  24836.4 ms  ✓ Highlights
  43317.2 ms  ✓ JuliaInterpreter
   3540.2 ms  ✓ Debugger
  4 dependencies successfully precompiled in 65 seconds. 462 already precompiled.


In [13]:
Pkg.add("ForwardDiff")

   Resolving package versions...
    Updating `~/.julia/environments/v1.11/Project.toml`
⌅ [f6369f11] + ForwardDiff v1.0.0
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


In [15]:
using Debugger
using SparseArrays
using Plots
using DelimitedFiles
using DataFrames
using LinearAlgebra
using JLD
using Optim
using Statistics
using Printf
using Plots.PlotMeasures
using ForwardDiff

In [16]:
default(; titlefontfamily = "Computer Modern",
    xguidefontfamily = "Computer Modern",
    yguidefontfamily = "Computer Modern",
    legendfontfamily = "Computer Modern",
    titlefontsize = 15,
    xguidefontsize = 12,
    legendfontsize = 12,
    yguidefontsize = 12,
    xgrid = :none)


In [ ]:
function bm_eql(fe)